In-situ Mixer Calibration
------------------------

The Cluster RF Modules (QCM-RF and QRM-RF) have integrated IQ mixers that handle both upconversion and downconversion of RF signals.
These mixers use a local oscillator (LO) with a frequency of $\omega_{LO}$.
During upconversion, they combine signals from the I and Q paths, which have a frequency of $\omega_{NCO}$, resulting in an output signal at $\omega_{LO} + \omega_{NCO}$.
However, due to the inherent mathematical imperfections of IQ mixers, the output also includes an unwanted signal at $\omega_{LO}$ (known as leakage) and another at $\omega_{LO} - \omega_{NCO}$ (called the undesired sideband).


In this tutorial, we are going to look at ways to suppress the LO leakage and undesired sideband. This process is called mixer calibration.

To run this tutorial optimally, you will need:

* A QCM-RF or QRM-RF module

* Spectrum analyzer

* Two SMA-cables

Setup
-----

First, we are going to import the required packages.

In [1]:
import json

In [2]:
from __future__ import annotations

from typing import TYPE_CHECKING, Callable

from qcodes.instrument import find_or_create_instrument

from qblox_instruments import Cluster, ClusterType

if TYPE_CHECKING:
    from qblox_instruments.qcodes_drivers.module import Module

### Scan For Clusters

We scan for the available devices connected via ethernet using the Plug & Play functionality of the Qblox Instruments package (see [Plug & Play](https://qblox-qblox-instruments.readthedocs-hosted.com/en/main/api_reference/tools.html#api-pnp) for more info).

In [3]:
!qblox-pnp list

Devices:
 - 192.168.137.2: cluster_mm 0.9.1 with name "cluster-mm" and serial number 00015_2251_003


In [4]:
cluster_ip = "192.168.137.2"
cluster_name = "cluster0"

### Connect to Cluster

We now make a connection with the Cluster.

In [5]:
cluster = find_or_create_instrument(
    Cluster,
    recreate=True,
    name=cluster_name,
    identifier=cluster_ip,
    dummy_cfg=(
        {
            2: ClusterType.CLUSTER_QCM,
            4: ClusterType.CLUSTER_QRM,
            6: ClusterType.CLUSTER_QCM_RF,
            8: ClusterType.CLUSTER_QRM_RF,
        }
        if cluster_ip is None
        else None
    ),
)

#### Get connected modules

In [6]:
def get_connected_modules(cluster: Cluster, filter_fn: Callable | None = None) -> dict[int, Module]:
    def checked_filter_fn(mod: ClusterType) -> bool:
        if filter_fn is not None:
            return filter_fn(mod)
        return True

    return {
        mod.slot_idx: mod for mod in cluster.modules if mod.present() and checked_filter_fn(mod)
    }

In [7]:
# QCM-RF modules
modules = get_connected_modules(cluster, lambda mod: not mod.is_qrm_type and mod.is_rf_type)

In [8]:
# This uses the module of the correct type with the lowest slot index
module = list(modules.values())[0]

In [9]:
module

<Module: cluster0_module6 of Cluster: cluster0>

### Reset the Cluster

We reset the Cluster to enter a well-defined state. Note that resetting will clear all stored parameters, so resetting between experiments is usually not desirable.

In [104]:
cluster.reset()
print(cluster.get_system_status())

Status: OKAY, Flags: NONE, Slot flags: NONE


We upload a simple sequence program that keeps playing the DC waveform at 30 percent IF power (waveform amplitude = 0.3). This will be modulated and upconverted within the QRM-RF before outputting.

In [105]:
# Sequence program.
seq_prog = """
      wait_sync 4

loop: play    0,0,1200
      jmp     @loop
"""
waveforms = {"dc": {"data": [0.3 for i in range(0, 1200)], "index": 0}}

# Add sequence to single dictionary and write to JSON file.
sequence = {
    "waveforms": waveforms,
    "weights": {},
    "acquisitions": {},
    "program": seq_prog,
}
with open("sequence.json", "w", encoding="utf-8") as file:
    json.dump(sequence, file, indent=4)
    file.close()

module.sequencer0.sequence("sequence.json")

Let's configure the sequencer to generate an IF frequency of $100$ MHz. To get an output frequency of $5.0$ GHz, we then have to configure the LO to run at $4.9$ GHz.

In [106]:
# Configure the Local oscillator

lo_freq = 4.9e9
nco_freq = 100e6
module.disconnect_outputs()

In [ ]:
# Configure channel map
module.sequencer0.connect_sequencer("out0")
module.out0_lo_freq(lo_freq)
# module.sequencer0.nco_freq(nco_freq)

In [110]:
module.sequencer0.marker_ovr_en(True)
module.sequencer0.marker_ovr_value(3)  # Enables output on QRM-RF

# Configure the sequencer
module.sequencer0.mod_en_awg(True)
module.sequencer0.nco_freq(nco_freq)
module.sequencer0.sync_en(True)

module.arm_sequencer(0)
module.start_sequencer(0)

print(module.get_sequencer_status(0))

Status: OKAY, State: RUNNING, Info Flags: NONE, Warning Flags: NONE, Error Flags: NONE, Log: []


Connect the output of the QRM-RF (O1) to the spectrum analyzer. This is what the output looks like on the spectrum analyzer (center frequency at 4.85 GHz with 600 MHz bandwidth). We see three peaks which correspond to (from the right) 5 GHz (desired signal), 4.9 GHz (LO Leakage) and 4.8 GHz (unwanted sideband).

![IQ_Mixer_Calib_before.png](figures/IQ_Mixer_Calib_before.png)

We will use the hardware mixer calibration capabilities of the Cluster RF modules to suppress the LO and sideband. This calibration routine is carried out by the internal circuitry of the modules hence requiring no cabling on the output channels.

**PLEASE NOTE**:

1. The output switches are turned OFF when the modules are performing calibration and are turned back ON after calibration.
2. Calibration also interrupts other sequencers within the module. Consequently, please restart the necessary sequencers (`arm_sequencer` and `start_sequencer`) that should be operational after completing the calibration process. Sequencers in other modules will continue to run as usual.
3. We expect a typical suppression of **35 dBc for LO and sideband tones when calibrating a (signal) tone with 30 percent of the maximum IF power**. It is possible to achieve more than 55 dBc of spurious free dynamic range (SFDR) when calibrating the mixers manually. Please follow the subsequent section for manually calibrating the mixer.

### Suppression of LO Leakage
The `module.out0_in0_lo_cal()` or `module.out{x}_lo_cal()` functions can be called to calibrate the LO leakage as shown below.

In [ ]:
module.out0_lo_cal()
module.sequencer0.sideband_cal()

### Suppression of Undesired Sideband
Similar to the LO suppression scheme, undesired sideband can also be suppressed using the `sequencer.sideband_cal()` function.

In [109]:
module.sequencer0.sideband_cal()

**NOTE** : If you would like to achieve better SFDR for your experiments, please calibrate the mixer manually.

Stop
----

Finally, let's stop the sequencers if they haven't already and close the instrument connection. One can also display a detailed snapshot containing the instrument parameters before
closing the connection by uncommenting the corresponding lines.

In [111]:
# Stop both sequencers.
module.stop_sequencer()

# Print status of both sequencers (should now say it is stopped).
print(module.get_sequencer_status(0))
print(module.get_sequencer_status(1))
print()

# Print an overview of the instrument parameters.
print("Snapshot:")
module.print_readable_snapshot(update=True)

# Reset the cluster
cluster.reset()
print(cluster.get_system_status())

Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []
Status: OKAY, State: STOPPED, Info Flags: NONE, Warning Flags: NONE, Error Flags: NONE, Log: []

Snapshot:
cluster0_module6:
	parameter                    value
--------------------------------------------------------------------------------
marker0_exp0_config           :	bypassed 
marker0_exp1_config           :	bypassed 
marker0_exp2_config           :	bypassed 
marker0_exp3_config           :	bypassed 
marker0_fir_config            :	bypassed 
marker0_inv_en                :	False 
marker1_exp0_config           :	bypassed 
marker1_exp1_config           :	bypassed 
marker1_exp2_config           :	bypassed 
marker1_exp3_config           :	bypassed 
marker1_fir_config            :	bypassed 
marker1_inv_en                :	False 
marker2_exp0_config           :	bypassed 
marker2_exp1_config           :	bypassed 
marker2_exp2_config           :	bypassed 
marker2_exp3_config           :